# catnip colab — stage 1 training (LEGACY / FALLBACK)

**Primary training host is Lightning AI Studio** (see `docs/r2-migration-plan.md`
Phase 4).  `notebooks/catnipKaggle.ipynb` is the second option.  Keep this
notebook for emergency fallback only.  Data lives in **Cloudflare R2**
(`catnip-canonical` bucket) — GCS is decommissioned.


### 0. R2 credentials


In [ ]:
try:
    import boto3
except ImportError:
    !pip install --quiet boto3
    import boto3

from google.colab import userdata

# R2 S3 API credentials (Cloudflare dashboard -> R2 -> API tokens).
# Add R2_ACCESS_KEY_ID / R2_SECRET_ACCESS_KEY as Colab secrets
# (the key-lock icon in the left sidebar).
R2_ENDPOINT = "https://eccb0b608bd926e793224944aba32cf4.r2.cloudflarestorage.com"
BUCKET = "catnip-canonical"

s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=userdata.get("R2_ACCESS_KEY_ID"),
    aws_secret_access_key=userdata.get("R2_SECRET_ACCESS_KEY"),
    region_name="auto",  # required by boto3, ignored by R2
)

# sanity: object count for the sliced payload
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix="training/stage1_sliced/", MaxKeys=1)
print("R2 reachable, KeyCount:", resp.get("KeyCount", 0))


### 1. pull data from R2 + install deps


In [ ]:
REPO_URL = "https://github.com/rifusaki/catnip.git"
REPO_DIR = "/content/catnip"
BRANCH = "refactor/reID"

# 1. Install dependencies first before pulling 14 GB payload
!pip install --quiet "ultralytics>=8.4.183,<9" "pydantic>=2.11,<3" "pydantic-settings>=2.11,<3" "omegaconf>=2.3,<3" "sahi>=0.11.36,<0.13" "opencv-python>=4.10,<5" "python-dotenv>=1,<2" "scikit-learn>=1.7,<2" "manga109api>=0.3.1,<0.4"

# 2. Clone repository
!git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

import os, sys, shutil
from pathlib import Path

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 3. Preflight disk check (requires >= 35 GB free for download + untar)
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"Colab /content free disk: {free_gb:.1f} GB")
assert free_gb >= 35.0, f"Insufficient disk space: {free_gb:.1f} GB < 35 GB required."

LOCAL_DATA_ROOT = "/content/catnip-data-local"
os.makedirs(f"{LOCAL_DATA_ROOT}/models", exist_ok=True)

# 4. Pull pre-sliced dataset archive from R2
print("Downloading dataset archive from R2 (14 GB)...")
s3.download_file(BUCKET, "compressed/stage1_sliced.tar", "/content/stage1_sliced.tar")

# 5. Extract to LOCAL_DATA_ROOT (tarball root is training/stage1_sliced/)
print("Extracting dataset...")
!tar -xf /content/stage1_sliced.tar -C {LOCAL_DATA_ROOT}
!rm /content/stage1_sliced.tar

# 6. Fetch pretrained base weights yolo26n.pt
print("Downloading base model weights yolo26n.pt from R2...")
s3.download_file(BUCKET, "models/yolo26n.pt", f"{LOCAL_DATA_ROOT}/models/yolo26n.pt")

# 7. Environment variable & integrity asserts
%env CATNIP_DATA={LOCAL_DATA_ROOT}
os.environ["CATNIP_DATA"] = LOCAL_DATA_ROOT

dataset_yaml_path = Path(f"{LOCAL_DATA_ROOT}/training/stage1_sliced/dataset.yaml")
base_model_path = Path(f"{LOCAL_DATA_ROOT}/models/yolo26n.pt")

assert dataset_yaml_path.exists(), f"Extraction error: {dataset_yaml_path} does not exist!"
assert base_model_path.exists(), f"Model fetch error: {base_model_path} does not exist!"

print("Data & base weights verified on local NVMe!")
!nvidia-smi -L || true
print(f"CATNIP_DATA={os.environ.get('CATNIP_DATA', 'UNSET')}")


### 2. train stage 1

Uses the pre-sliced dataset pulled from R2. Overrides `device` to `cuda` (config defaults to `mps` for local mac).

Defaults from `config/pipeline.yaml`: yolo26n, imgsz=640, batch=8, epochs=300, workers=4. Overridden for Colab nano test: batch=128, epochs=30.


In [ ]:
import sys, os, time, threading
from pathlib import Path
import yaml as _yaml

REPO_DIR = "/content/catnip"
os.chdir(REPO_DIR)
LOCAL_DATA_ROOT = "/content/catnip-data-local"
os.environ.setdefault("CATNIP_DATA", LOCAL_DATA_ROOT)

# --- Out-of-order / re-run safety: guarantee s3 + BUCKET exist even if
# the credentials cell (step 0) has not run yet in this session ---
if "BUCKET" not in globals():
    BUCKET = "catnip-canonical"
if "s3" not in globals():
    import boto3
    R2_ENDPOINT = os.environ.get(
        "R2_ENDPOINT",
        "https://eccb0b608bd926e793224944aba32cf4.r2.cloudflarestorage.com",
    )
    _r2_key = os.environ.get("R2_ACCESS_KEY_ID")
    _r2_secret = os.environ.get("R2_SECRET_ACCESS_KEY")
    if _r2_key is None or _r2_secret is None:
        try:
            from google.colab import userdata
            _r2_key = _r2_key or userdata.get("R2_ACCESS_KEY_ID")
            _r2_secret = _r2_secret or userdata.get("R2_SECRET_ACCESS_KEY")
        except ImportError:
            pass  # not on Colab; rely on env vars
    s3 = boto3.client(
        "s3",
        endpoint_url=R2_ENDPOINT,
        aws_access_key_id=_r2_key,
        aws_secret_access_key=_r2_secret,
        region_name="auto",  # required by boto3, ignored by R2
    )

print(f"CATNIP_DATA = {os.environ.get('CATNIP_DATA')}")
print(f"python = {sys.executable}")

# Patch dataset.yaml path to absolute root on Colab
_yaml_path = Path(f"{LOCAL_DATA_ROOT}/training/stage1_sliced/dataset.yaml")
_data = _yaml.safe_load(_yaml_path.read_text())
_data["path"] = str(_yaml_path.parent.resolve())
_yaml_path.write_text(_yaml.dump(_data, default_flow_style=False))
print(f"dataset.yaml path -> {_data['path']}")

# Start background thread to push checkpoints to R2 every 10 min (guards against preemption)
stop_watcher = threading.Event()
def checkpoint_watcher():
    last_pt = Path(f"{LOCAL_DATA_ROOT}/runs/detect/stage1/weights/last.pt")
    last_mtime = 0
    while not stop_watcher.is_set():
        if last_pt.exists():
            try:
                mtime = last_pt.stat().st_mtime
                if mtime > last_mtime:
                    s3.upload_file(str(last_pt), BUCKET, "runs/stage1-last.pt")
                    print(f"\n[R2 Checkpoint] Pushed last.pt to s3://{BUCKET}/runs/stage1-last.pt")
                    last_mtime = mtime
            except Exception as e:
                print(f"\n[R2 Checkpoint Warning] Failed to push checkpoint: {e}")
        stop_watcher.wait(600)  # Check every 10 minutes

watcher_thread = threading.Thread(target=checkpoint_watcher, daemon=True)
watcher_thread.start()

# Run training in-process with sys.argv restoration
_orig_argv = sys.argv.copy()
sys.argv = [
    "stage1.py",
    "--override", "training.stage1.device=cuda",
    "--override", "training.stage1.workers=8",       # High-RAM provides 8-12 vCPUs
    "--override", "training.stage1.batch=64",        # Fits easily in L4 (24GB) or A100 (40GB)
    "--override", "training.stage1.epochs=100",      # Full convergence schedule
    "--override", "training.stage1.use_subset=false",# Train on all 132k slices!
    "--verbose",
]

from scripts.train import stage1
try:
    stage1.main()
except SystemExit as e:
    if e.code != 0:
        print(f"\nTraining failed with exit code {e.code}.")
        raise
finally:
    stop_watcher.set()
    sys.argv = _orig_argv


### 3. verify

In [ ]:
import os, sys
from pathlib import Path

REPO_DIR = "/content/catnip"
os.chdir(REPO_DIR)
LOCAL_DATA_ROOT = "/content/catnip-data-local"
os.environ.setdefault("CATNIP_DATA", LOCAL_DATA_ROOT)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.config import load_settings

settings = load_settings("config/pipeline.yaml")
best_model = settings.paths.model_dir / "yolo26_stage1_body_head_face.pt"
runs_dir = settings.paths.runs_dir / "detect" / "stage1"

print(f"best model: {best_model}  {'OK' if best_model.exists() else 'MISSING'}")
print(f"runs dir:   {runs_dir}")

results_csv = runs_dir / "results.csv"
if results_csv.exists():
    print(f"\nlast 5 epochs:")
    !tail -5 {results_csv}

results_png = runs_dir / "results.png"
if results_png.exists():
    print(f"curves: {results_png}")


### 4. upload results to R2


In [ ]:
print("Uploading final model and runs to R2...")
from pathlib import Path

LOCAL_DATA_ROOT = "/content/catnip-data-local"
MODEL = Path(f"{LOCAL_DATA_ROOT}/models/yolo26_stage1_body_head_face.pt")
RUNS_DIR = Path(f"{LOCAL_DATA_ROOT}/runs/detect/stage1")

if MODEL.exists():
    s3.upload_file(str(MODEL), BUCKET, "models/yolo26_stage1_body_head_face.pt")
    print(f"Deliverable model uploaded to s3://{BUCKET}/models/yolo26_stage1_body_head_face.pt")
else:
    print(f"Warning: best model not found at {MODEL}")

if RUNS_DIR.exists():
    !tar -cf /content/stage1-run.tar -C {LOCAL_DATA_ROOT} runs/detect/stage1
    s3.upload_file("/content/stage1-run.tar", BUCKET, "runs/stage1-run.tar")
    !rm /content/stage1-run.tar
    print(f"Run directory archive uploaded to s3://{BUCKET}/runs/stage1-run.tar")

print("Done!")
